# Why use a stochastic gradient descent over batch gradient descent?

The batch gradient is suitable for small datasets since the matrices are formed and lot of computation is required.
The batch gradient decent might end up in the local minima but in stochastic gradient descent you'll end up finding the global minima
but stochastic is less stable algo than batch gradient descent 

# But how does this stochastic gradient descent work?

just instead of a y_hat matrix we calculate the y_hat on each row which is randomized with the range of 0 - number of rows in X_train 

1️⃣ Normal Equation (Closed-form solution)
Formula:

𝜃
=
(
𝑋
𝑇
𝑋
)
−
1
𝑋
𝑇
𝑦
θ=(X 
T
 X) 
−1
 X 
T
 y
When to use:

✅ Small to medium datasets (features < a few thousand, samples up to maybe ~10k–50k depending on hardware).

✅ You want exact solution (no approximation).

✅ Features are not too many — computing 
(
𝑋
𝑇
𝑋
)
−
1
(X 
T
 X) 
−1
  is O(n³) in complexity, which becomes slow and memory-heavy for very large 
𝑛
n.

✅ You don’t need to tune learning rates or epochs.

❌ Not good for huge datasets — memory can explode.

2️⃣ SGD Regressor (Stochastic Gradient Descent)
This is an iterative optimization:

Updates parameters using one sample (or mini-batch) at a time.

Approximates the solution rather than computing it exactly.

When to use:

✅ Large datasets that don’t fit into memory — can process data in batches or streams.

✅ Online learning — when new data arrives continuously and you want to update the model without retraining from scratch.

✅ When features are very large (e.g., text data with tens of thousands of features from bag-of-words/TF-IDF).

✅ If you need fast training for high-dimensional sparse datasets.

❌ Requires hyperparameter tuning (learning rate, number of epochs, regularization).

❌ Not guaranteed to hit the exact solution (but can get very close).



In [85]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np
from sklearn.linear_model import LinearRegression

In [86]:
X,y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
r2_score(y_test, y_pred)

0.45260276297191926

In [87]:
X

array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
         0.01990749, -0.01764613],
       [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
        -0.06833155, -0.09220405],
       [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
         0.00286131, -0.02593034],
       ...,
       [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
        -0.04688253,  0.01549073],
       [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
         0.04452873, -0.02593034],
       [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
        -0.00422151,  0.00306441]])

In [88]:
print(lr.coef_, lr.intercept_)

[  37.90402135 -241.96436231  542.42875852  347.70384391 -931.48884588
  518.06227698  163.41998299  275.31790158  736.1988589    48.67065743] 151.34560453985995


In [89]:
class SGDGradientDescent:
    def __init__(self, lr=0.01, num_iter=1000):
        self.lr = lr
        self.num_iter = num_iter
        self.coef_ = None
        self.intercept_ = None
        #x_train.shape will give us the number of coefficients we need to calculate


    def fit(self,X,y):
        self.intercept_ = 0
        self.coef_ = np.ones(X.shape[1]) #number of columns in X

        for i in range(self.num_iter):
            for j in range(X.shape[0]): #number of rows in X
                # Predict the value for the j-th sample
                idx = np.random.randint(0, X.shape[0]) #generates a random index within the range 
                y_hat = self.intercept_ + np.dot(X[idx], self.coef_) #since we are using a single sample, we don't need to average and calculate for a single index

                #intercept and coefficients will have the same formula as batch gradient descent 
                intercept_derivative = 2 * (y[idx] - y_hat)
                self.intercept_ += self.lr * intercept_derivative

                coef_derivative = 2 * np.dot((y[idx] - y_hat),X[idx]) #/ X.shape[0] we dont use this because this will always be 1
                self.coef_ += self.lr * coef_derivative



    def predict(self,X):
        return self.intercept_ + np.dot(X, self.coef_)

In [ ]:
sgd = SGDGradientDescent(lr=0.01, num_iter=3)
sgd.fit(X_train, y_train)
y_pred = sgd.predict(X_test)
print(sgd.coef_, sgd.intercept_)

[  42.99658645 -240.04916362  550.28829638  340.48012218  -97.76613973
 -125.69623442 -215.45476514  149.3719548   414.80447479   66.17598918] 151.4904020971111
